# Log-mel Representation Features (IEMOCAP)

This notebook extracts pooled log-mel spectrogram summaries.
Each utterance becomes one training row for downstream SER models.

In [1]:
from pathlib import Path

import librosa
import numpy as np
import pandas as pd


In [2]:
# Configuration
REPO_ROOT = Path.cwd().parents[1]  # repo root (notebook is under feature_extraction/)
CSV_PATH = REPO_ROOT / "datasets" / "IEMOCAP" / "iemocap_full_dataset.csv"
AUDIO_ROOT = REPO_ROOT / "datasets" / "IEMOCAP"
OUT_DIR = REPO_ROOT / "extracted_features" / "representations"
OUT_FILE = "representations_features.csv"

# Audio + feature params
TARGET_SR = 16_000
N_FFT = 1024
HOP_LENGTH = 256
N_MELS = 64
FMIN = 50
FMAX = None  # defaults to sr // 2

OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / OUT_FILE
OUT_PATH


In [3]:
def load_audio(path: Path) -> tuple[np.ndarray, int]:
    # Load audio and resample to TARGET_SR so features are comparable
    audio, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    return audio, sr


def compute_log_mel(audio: np.ndarray, sr: int) -> np.ndarray:
    # Compute log-mel spectrogram
    fmax = sr // 2 if FMAX is None else FMAX
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=fmax,
        power=2.0,
    )
    return librosa.power_to_db(mel, ref=np.max)


def summarize_matrix(prefix: str, matrix: np.ndarray) -> dict[str, float]:
    # Summary stats across all values in the matrix
    return {
        f"{prefix}_mean": float(matrix.mean()),
        f"{prefix}_std": float(matrix.std()),
        f"{prefix}_min": float(matrix.min()),
        f"{prefix}_max": float(matrix.max()),
        f"{prefix}_median": float(np.median(matrix)),
    }


def extract_representation_features(audio: np.ndarray, sr: int) -> dict[str, float]:
    log_mel = compute_log_mel(audio, sr)
    features: dict[str, float] = {
        "log_mel_frames": float(log_mel.shape[1]),
        "log_mel_bands": float(log_mel.shape[0]),
    }
    features.update(summarize_matrix("log_mel_db", log_mel))

    band_means = log_mel.mean(axis=1)
    band_stds = log_mel.std(axis=1)
    for idx, (mean_val, std_val) in enumerate(zip(band_means, band_stds)):
        features[f"mel_band{idx:02d}_mean_db"] = float(mean_val)
        features[f"mel_band{idx:02d}_std_db"] = float(std_val)
    return features


In [4]:
df = pd.read_csv(CSV_PATH)  # metadata for paths + labels
df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()

# Filter: valid emotion + agreement > 0
df = df[(df["emotion"] != "xxx") & (df["agreement"] > 0)].copy()
df.shape


In [5]:
rows: list[dict[str, float | str | int]] = []
missing: list[str] = []

for _, row in df.iterrows():
    rel_path = row["path"]
    audio_path = AUDIO_ROOT / rel_path
    if not audio_path.exists():
        missing.append(str(audio_path))
        continue

    audio, sr = load_audio(audio_path)
    duration_s = audio.shape[0] / sr
    features = extract_representation_features(audio, sr)

    record: dict[str, float | str | int] = {
        "path": str(rel_path),
        "session": int(row["session"]),
        "method": row["method"],
        "gender": row["gender"],
        "emotion": row["emotion"],
        "n_annotators": int(row["n_annotators"]),
        "agreement": int(row["agreement"]),
        "duration_s": float(duration_s),
    }
    record.update(features)
    rows.append(record)

feature_df = pd.DataFrame(rows)
feature_df.to_csv(OUT_PATH, index=False)

print(f"Saved: {OUT_PATH}")
if missing:
    print(f"Missing audio files: {len(missing)}")
feature_df.shape
